# 04 - Modelling
## 01. Data loading and feature setup

In [5]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/processed/hourly_demand_weather.csv', parse_dates=['date'])

TARGET = 'trips'
TIME = ['hour_sin', 'hour_cos', 'is_weekend', 'is_holiday']
LAGS = ['lag_24', 'lag_168', 'roll_24']
WEATHER = ['temperature_2m', 'precipitation', 'wind_speed_10m', 'relative_humidity_2m', 'is_rain']

use = df[df['exclude'] == 0]
train = use[use['split'] == 'train']
val = use[use['split'] == 'val']
test = use[use['split'] == 'test']
print(len(train), len(val), len(test))

10058 2232 2304


## 2. Evaluation function

In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(y_true, y_pred): # all models are compared with the same metrics (MAE, RMSE, R²). Predictions are clipped at zero because trips cannot be negative
    y_pred = np.clip(y_pred, 0, None)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }

## 3. Model 0: baseline (same hour last week)

In [7]:
for name, part in [('val', val), ('test', test)]: # The simplest forecast sets the minimum accuracy that all models must beat
    print(name, evaluate(part[TARGET], part['lag_168']))

val {'MAE': 8.376792114695341, 'RMSE': np.float64(13.5033349126592), 'R2': 0.6166396580855378}
test {'MAE': 7.081597222222222, 'RMSE': np.float64(11.899244257047961), 'R2': 0.766407353231628}


## 4. Model 1: Ridge regression
Simple, interpretable linear model. City is one-hot encoded and numeric features are scaled inside the pipeline, fitted on the training set only. Feature groups are added step by step (time, lags, weather).

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

def make_ridge(num_cols, alpha):
    prep = ColumnTransformer([
        ('city', OneHotEncoder(handle_unknown='ignore'), ['city']),
        ('num', StandardScaler(), num_cols),
    ])
    return Pipeline([('prep', prep), ('model', Ridge(alpha=alpha))])

In [10]:
sets = {'time': TIME, 'time+lags': TIME + LAGS, 'time+lags+weather': TIME + LAGS + WEATHER}
rows = []
for name, cols in sets.items():
    for a in [0.1, 1, 10, 100]:
        m = make_ridge(cols, a).fit(train[['city'] + cols], train[TARGET])
        res = evaluate(val[TARGET], m.predict(val[['city'] + cols]))
        rows.append({'model': 'Ridge', 'features': name, 'alpha': a, **res})

ridge_val = pd.DataFrame(rows).round(2)
ridge_val

,model,features,alpha,MAE,RMSE,R2
0,Ridge,time,0.1,13.69,18.96,0.24
1,Ridge,time,1.0,13.69,18.95,0.24
2,Ridge,time,10.0,13.67,18.91,0.25
3,Ridge,time,100.0,13.53,18.51,0.28
4,Ridge,time+lags,0.1,6.53,10.06,0.79
5,Ridge,time+lags,1.0,6.53,10.06,0.79
6,Ridge,time+lags,10.0,6.53,10.07,0.79
7,Ridge,time+lags,100.0,6.53,10.09,0.79
8,Ridge,time+lags+weather,0.1,6.89,10.19,0.78
9,Ridge,time+lags+weather,1.0,6.89,10.19,0.78


## 5. Model 2: Random Forest
Captures non-linear pattern such as the two daily peaks and interactions between hour, weekday and level of demand

In [11]:
from sklearn.ensemble import RandomForestRegressor

def make_rf(num_cols, max_depth, min_leaf):
    prep = ColumnTransformer([
        ('city', OneHotEncoder(handle_unknown='ignore'), ['city']),
        ('num', 'passthrough', num_cols),
    ])
    rf = RandomForestRegressor(n_estimators=200, max_depth=max_depth,
                               min_samples_leaf=min_leaf, n_jobs=-1, random_state=42)
    return Pipeline([('prep', prep), ('model', rf)])

In [12]:
rows = []
for name, cols in sets.items():
    for depth in [8, None]:
        for leaf in [5, 20]:
            m = make_rf(cols, depth, leaf).fit(train[['city'] + cols], train[TARGET])
            res = evaluate(val[TARGET], m.predict(val[['city'] + cols]))
            rows.append({'model': 'Random forest', 'features': name,
                         'max_depth': depth, 'min_leaf': leaf, **res})

rf_val = pd.DataFrame(rows).round(2)
rf_val

,model,features,max_depth,min_leaf,MAE,RMSE,R2
0,Random forest,time,8.0,5,11.62,17.61,0.35
1,Random forest,time,8.0,20,11.65,17.59,0.35
2,Random forest,time,NaN,5,11.31,17.36,0.37
3,Random forest,time,NaN,20,11.42,17.40,0.36
4,Random forest,time+lags,8.0,5,6.54,10.31,0.78
5,Random forest,time+lags,8.0,20,6.45,10.13,0.78
6,Random forest,time+lags,NaN,5,6.60,10.51,0.77
7,Random forest,time+lags,NaN,20,6.38,10.09,0.79
8,Random forest,time+lags+weather,8.0,5,6.49,10.14,0.78
9,Random forest,time+lags+weather,8.0,20,6.46,10.06,0.79


In [14]:
cols = TIME + LAGS + WEATHER
best_rf = make_rf(cols, None, 20).fit(train[['city'] + cols], train[TARGET])
pred = np.clip(best_rf.predict(val[['city'] + cols]), 0, None)

err = (pred - val[TARGET])
print('Mean error (prediction minus actual):', err.mean().round(2))
print(err.groupby(val['city']).mean().round(2))

Mean error (prediction minus actual): -1.49
city
Badalona        -2.37
Brno             0.80
Frankfurt       -3.44
Frydek-Mistek    1.61
Havirov          0.18
Leon            -2.34
Opava           -0.87
Ostrava         -6.31
Name: trips, dtype: float64


In [17]:
chk = pd.DataFrame({
    'train_mean': train.groupby('city')[TARGET].mean(),
    'val_mean': val.groupby('city')[TARGET].mean(),
    'bias': err.groupby(val['city']).mean(),
}).round(2)
chk['bias_pct_of_val_mean'] = (chk['bias'] / chk['val_mean'] * 100).round(1)
print(chk)

               train_mean  val_mean  bias  bias_pct_of_val_mean
city                                                           
Badalona            24.43     29.98 -2.37                  -7.9
Brno                24.98     14.41  0.80                   5.6
Frankfurt           14.05     22.90 -3.44                 -15.0
Frydek-Mistek       18.12      7.19  1.61                  22.4
Havirov             18.89     11.14  0.18                   1.6
Leon                16.77     21.95 -2.34                 -10.7
Opava               19.90     15.65 -0.87                  -5.6
Ostrava             64.99     40.37 -6.31                 -15.6


## 6. Model 3: Gradient boosting
Builds trees one after another, each correcting the errors of the previous ones. It is usually the most accurate model on tabular data, but it is more sensitive to settings.

In [15]:
from sklearn.ensemble import HistGradientBoostingRegressor

def make_gb(num_cols, lr, leaves):
    prep = ColumnTransformer([
        ('city', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['city']),
        ('num', 'passthrough', num_cols),
    ])
    gb = HistGradientBoostingRegressor(learning_rate=lr, max_leaf_nodes=leaves,
                                       max_iter=300, l2_regularization=1.0, random_state=42)
    return Pipeline([('prep', prep), ('model', gb)])

In [16]:
rows = []
for name, cols in sets.items():
    for lr in [0.03, 0.1]:
        for leaves in [15, 31]:
            m = make_gb(cols, lr, leaves).fit(train[['city'] + cols], train[TARGET])
            res = evaluate(val[TARGET], m.predict(val[['city'] + cols]))
            rows.append({'model': 'Gradient boosting', 'features': name,
                         'learning_rate': lr, 'max_leaf_nodes': leaves, **res})

gb_val = pd.DataFrame(rows).round(2)
gb_val

,model,features,learning_rate,max_leaf_nodes,MAE,RMSE,R2
0,Gradient boosting,time,0.03,15,11.59,17.57,0.35
1,Gradient boosting,time,0.03,31,11.46,17.51,0.36
2,Gradient boosting,time,0.10,15,11.52,17.57,0.35
3,Gradient boosting,time,0.10,31,11.43,17.51,0.36
4,Gradient boosting,time+lags,0.03,15,6.63,10.58,0.76
5,Gradient boosting,time+lags,0.03,31,6.76,10.87,0.75
6,Gradient boosting,time+lags,0.10,15,6.72,10.86,0.75
7,Gradient boosting,time+lags,0.10,31,6.84,11.06,0.74
8,Gradient boosting,time+lags+weather,0.03,15,6.21,9.81,0.80
9,Gradient boosting,time+lags+weather,0.03,31,6.34,10.09,0.79
